# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook chạy được trên **Colab T4** và **Kaggle (GPU T4)**: trên Kaggle, gắn dataset `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes` và bật Internet; notebook tự clone `scripts/` + `configs/` của đề. Mọi bước huấn luyện/suy luận đều **chạy tiếp được** nếu bị ngắt.

## 0. Môi trường (Colab T4 / Kaggle T4)

In [ ]:
# Chạy được trên Colab (Runtime → T4 GPU) và Kaggle (Accelerator GPU T4, bật Internet,
# gắn dataset lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes làm input).
import os, sys, subprocess, time
T_START = time.time()
IS_KAGGLE = os.path.isdir("/kaggle/input")
IS_COLAB = "google.colab" in sys.modules
SMOKE = False   # True: chạy thử cả pipeline với vài ảnh / vài bước để bắt lỗi trước khi chạy thật
REPO_URL = "https://github.com/AIVIETNAM-AIO-AnhDinh/K4-Track-4-Day10-VisionLanguageModel.git"

if IS_KAGGLE and not os.path.exists("scripts/score.py"):
    # Trên Kaggle notebook chạy một mình: lấy scripts/ và configs/ của đề từ repo
    os.chdir("/kaggle/working")
    if not os.path.exists("lab/scripts/score.py"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "lab"], check=True)
    os.chdir("lab")
sys.path.insert(0, os.getcwd())
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("cwd:", os.getcwd(), "| kaggle" if IS_KAGGLE else "| colab" if IS_COLAB else "| local", "| SMOKE" if SMOKE else "")
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
# Ảnh Kaggle cài sẵn torchao 0.10: peft 0.21 thấy torchao/gptqmodel bản cũ là raise ImportError ngay khi gắn LoRA
# (không bỏ qua). Notebook không dùng hai gói này → gỡ đi, rồi kiểm tra sớm thay vì đợi tới sau E0.
!pip -q uninstall -y torchao gptqmodel
subprocess.run([sys.executable, "-c", "from peft.import_utils import is_torchao_available as a, is_gptqmodel_available as b; a(); b()"],
               check=True)

In [ ]:
# Kaggle API key chỉ cần khi tải dataset bằng Kaggle CLI (Colab). KHÔNG viết khoá vào notebook.
# Trên Kaggle, dataset đã được gắn sẵn vào /kaggle/input nên không cần khoá.
if IS_COLAB:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
def sh(*cmd):
    """Chạy lệnh, in output và DỪNG nếu lệnh lỗi (khác `!cmd`, vốn lặng lẽ bỏ qua lỗi)."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    print(r.stdout[-6000:], r.stderr[-4000:] if r.returncode else "", sep="")
    r.check_returncode()
    return r.stdout

if IS_KAGGLE:
    sh(sys.executable, "scripts/prepare_data.py", "--source", "/kaggle/input")
else:
    sh(sys.executable, "scripts/prepare_data.py", "--kaggle")

In [ ]:
import gc, json, math, random, shutil, traceback
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from scripts.labcommon import CLASSES, read_jsonl, write_jsonl

DATA = Path("data")
splits = {s: read_jsonl(DATA / "splits" / f"{s}.jsonl") for s in ("train", "val", "test")}
META = json.loads((DATA / "splits" / "meta.json").read_text(encoding="utf-8"))
print("dataset_version:", META["dataset_version"])
for s, rows in splits.items():
    boxes = {c: sum(o["label"] == c for r in rows for o in r["objects"]) for c in CLASSES}
    print(f"{s:5s} {len(rows):5d} ảnh  {len({r['session'] for r in rows}):3d} phiên  "
          f"ảnh không box={sum(not r['objects'] for r in rows):3d}  boxes={boxes}")
sess = {s: {r["session"] for r in rows} for s, rows in splits.items()}
assert not (sess["train"] & sess["val"] or sess["train"] & sess["test"] or sess["val"] & sess["test"])
print("OK: không phiên quay nào xuất hiện ở hai split")

WORK = Path("work/smoke") if SMOKE else Path("work")
EVID = WORK / "evidence" if SMOKE else Path("report/evidence")
FIG = WORK / "figures" if SMOKE else Path("report/figures")
for p in (WORK, EVID, FIG):
    p.mkdir(parents=True, exist_ok=True)

COLORS = {"pothole": "red", "crack": "yellow", "manhole": "cyan"}

def load_image(entry):
    return Image.open(DATA / entry["image_path"]).convert("RGB")

def gt_pixel(entry):
    """Box gốc YOLO → [(label, (x1, y1, x2, y2))] theo pixel."""
    W, H = entry["width"], entry["height"]
    return [(o["label"], ((o["yolo"][0] - o["yolo"][2] / 2) * W, (o["yolo"][1] - o["yolo"][3] / 2) * H,
                          (o["yolo"][0] + o["yolo"][2] / 2) * W, (o["yolo"][1] + o["yolo"][3] / 2) * H))
            for o in entry["objects"]]

def draw_boxes(ax, boxes, color=None, ls="-", lw=1.8, text=True):
    for lab, (x1, y1, x2, y2) in boxes:
        c = color or COLORS.get(lab, "white")
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=c, lw=lw, ls=ls))
        if text:
            ax.text(x1, y1 - 2, lab, color=c, fontsize=7, va="bottom")

random.seed(0)
fig, axes = plt.subplots(2, 3, figsize=(15, 6))
for ax, e in zip(axes.flat, random.sample([r for r in splits["train"] if r["objects"]], 6)):
    ax.imshow(load_image(e)); draw_boxes(ax, gt_pixel(e)); ax.set_title(e["image_id"], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "train_samples.png", dpi=110); plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
def make_prompt(coord="norm1000"):
    scale = ("integers on a 0-1000 scale relative to the image width and height "
             "(0 = left/top edge, 1000 = right/bottom edge)" if coord == "norm1000"
             else "integer pixel coordinates of the 640x360 image")
    return ("Locate every road damage in this dashcam image. Classes: pothole, crack, manhole.\n"
            'Answer with ONLY a JSON list and no other text: [{"label": "<class>", "bbox_2d": [x1, y1, x2, y2]}, ...]\n'
            f"bbox_2d is the box (top-left x1,y1 and bottom-right x2,y2) in {scale}.\n"
            "If there is no road damage, answer [].")

PROMPT = make_prompt("norm1000")

def yolo_to_box(label, yolo, coord="norm1000", width=640, height=360):
    xc, yc, w, h = yolo
    sx, sy = (1000, 1000) if coord == "norm1000" else (width, height)
    def span(c, s, scale):
        lo = min(max(round((c - s / 2) * scale), 0), scale)
        hi = min(max(round((c + s / 2) * scale), 0), scale)
        if hi <= lo:  # box rất nhỏ: score.parse_output loại cả câu trả lời nếu x2 <= x1
            lo, hi = (lo, lo + 1) if lo < scale else (scale - 1, scale)
        return lo, hi
    x1, x2 = span(xc, w, sx)
    y1, y2 = span(yc, h, sy)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects, coord="norm1000", width=640, height=360):
    boxes = [yolo_to_box(o["label"], o["yolo"], coord, width, height) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))  # trên→dưới, trái→phải
    return json.dumps(boxes, separators=(",", ":"))

print(PROMPT)
e = max(splits["train"][:200], key=lambda r: len(r["objects"]))
ans = objects_to_answer(e["objects"])
print("\n", e["image_id"], ans)

# Kiểm tra: JSON thang 0–1000 → pixel phải trùng box gốc (đã kẹp vào khung ảnh; làm tròn lệch ≤ 0,32 px)
back = [(b["label"], (b["bbox_2d"][0] * 0.64, b["bbox_2d"][1] * 0.36, b["bbox_2d"][2] * 0.64, b["bbox_2d"][3] * 0.36))
        for b in json.loads(ans)]
err = 0.0
for o, (_, p) in zip(e["objects"], gt_pixel(e)):
    b = yolo_to_box(o["label"], o["yolo"])["bbox_2d"]
    q = (b[0] * 0.64, b[1] * 0.36, b[2] * 0.64, b[3] * 0.36)
    p = (min(max(p[0], 0), 640), min(max(p[1], 0), 360), min(max(p[2], 0), 640), min(max(p[3], 0), 360))
    err = max(err, *(abs(x - y) for x, y in zip(p, q)))
print(f"sai số lớn nhất khi quy ngược về pixel: {err:.2f} px")
assert err <= 0.33

# Mọi đáp án phải qua được parser của trọng tài và giữ nguyên số box
from scripts.score import parse_output
for s, rows in splits.items():
    for r in rows:
        for coord in ("norm1000", "pixel"):
            p = parse_output(objects_to_answer(r["objects"], coord))
            assert p is not None and len(p) == len(r["objects"]), (s, r["image_id"], coord)
print("OK: mọi đáp án JSON hợp lệ với scripts/score.py")

fig, ax = plt.subplots(1, 1, figsize=(8, 4.5))
ax.imshow(load_image(e)); draw_boxes(ax, gt_pixel(e), color="lime", lw=3, text=False); draw_boxes(ax, back, ls="--")
ax.set_title("gốc (xanh lá) vs JSON 0–1000 quy ngược (nét đứt)"); ax.axis("off"); plt.show()

## 3. E0: zero-shot

In [ ]:
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
TEST = splits["test"][:16] if SMOKE else splits["test"]
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"  # sinh theo batch: đệm trái để mọi prompt kết thúc cùng vị trí

def load_base():
    # ViT (gồm cả projector) và lm_head giữ fp16. Ở transformers 5.x mẫu skip được so khớp từ ĐẦU tên module
    # (re.match), nên phải ghi "model.visual": chỉ ghi "visual" thì ViT vẫn bị lượng tử hoá 4-bit.
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
                               llm_int8_skip_modules=["model.visual", "lm_head"])
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, dtype=torch.float16, device_map={"": 0})
    vit = model.model.visual
    assert type(vit.blocks[0].attn.qkv) is torch.nn.Linear, "ViT bị lượng tử hoá"
    assert type(vit.merger.linear_fc1) is torch.nn.Linear, "projector bị lượng tử hoá"
    assert type(model.model.language_model.layers[0].mlp.up_proj) is not torch.nn.Linear, "LLM chưa lượng tử hoá"
    return model

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def user_msg(image, prompt=PROMPT):
    return {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}

@torch.inference_mode()
def predict(model, entries, out_path, prompt=PROMPT, batch_size=8, max_new_tokens=256):
    """Sinh greedy, ghi {"image_id","raw"} mỗi dòng. Bị ngắt giữa chừng thì chạy lại sẽ làm tiếp."""
    out_path = Path(out_path)
    done = {r["image_id"] for r in read_jsonl(out_path)} if out_path.exists() else set()
    todo = [e for e in entries if e["image_id"] not in done]
    model.eval()
    t0 = time.time()
    with open(out_path, "a", encoding="utf-8") as f:
        for i in range(0, len(todo), batch_size):
            batch = todo[i:i + batch_size]
            inputs = processor.apply_chat_template(
                [[user_msg(load_image(e), prompt)] for e in batch], tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", padding=True).to(model.device)
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
            raws = processor.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
            for e, raw in zip(batch, raws):
                f.write(json.dumps({"image_id": e["image_id"], "raw": raw}, ensure_ascii=False) + "\n")
            f.flush()
            if (i // batch_size) % 10 == 0:
                print(f"  {i + len(batch)}/{len(todo)} ảnh, {time.time() - t0:.0f}s")
    print(f"{out_path}: {len(todo)} ảnh trong {(time.time() - t0) / 60:.1f} phút")

def score(key, name, pred=None, coord="norm1000", meta=None):
    cmd = [sys.executable, "scripts/score.py", "--pred", pred or WORK / f"{key}.jsonl", "--name", name,
           "--out", EVID / f"{key}.json", "--coord", coord]
    for k, v in (meta or {}).items():
        cmd += ["--meta", f"{k}={v}"]
    sh(*cmd)
    return json.loads((EVID / f"{key}.json").read_text(encoding="utf-8"))

In [ ]:
model = load_base()
print(f"VRAM sau khi nạp model 4-bit: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
predict(model, TEST, WORK / "e0.jsonl", max_new_tokens=256)
del model; free_gpu()
for r in read_jsonl(WORK / "e0.jsonl")[:5]:
    print(r["image_id"], repr(r["raw"][:400]))

In [ ]:
RESULTS = {"e0": score("e0", "E0")}

# Phân loại lỗi của E0 (cho câu 2)
raws = read_jsonl(WORK / "e0.jsonl")
parsed = [parse_output(r["raw"]) for r in raws]
ok = [p for p in parsed if p is not None]
boxes = [b for p in ok for b in p]
print(f"bọc ```json: {sum('```' in r['raw'] for r in raws)}/{len(raws)}"
      f" | JSON hỏng: {sum(p is None for p in parsed)}"
      f" | trả []: {sum(p == [] for p in ok)}"
      f" | box có toạ độ > 1000: {sum(max(b['bbox']) > 1000 for b in boxes)}/{len(boxes)}"
      f" | nhãn ngoài 3 lớp: {sum(b['label'] not in CLASSES for b in boxes)}"
      f" | số box TB/ảnh: {len(boxes) / max(len(ok), 1):.2f} (đúng: {sum(len(e['objects']) for e in TEST) / len(TEST):.2f})")

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
import torch.nn.functional as F

IM_END = processor.tokenizer.convert_tokens_to_ids("<|im_end|>")
IMAGE_TOKEN = processor.tokenizer.convert_tokens_to_ids(processor.image_token)
PAD_VALUES = {"input_ids": processor.tokenizer.pad_token_id, "attention_mask": 0, "mm_token_type_ids": 0, "labels": -100}

def encode_example(entry, prompt=PROMPT, coord="norm1000"):
    user = user_msg(load_image(entry), prompt)
    answer = objects_to_answer(entry["objects"], coord, entry["width"], entry["height"])
    assistant = {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    kw = dict(tokenize=True, return_dict=True, return_tensors="pt")
    prompt_enc = processor.apply_chat_template([user], add_generation_prompt=True, **kw)
    full_enc = processor.apply_chat_template([user, assistant], add_generation_prompt=False, **kw)
    n = prompt_enc["input_ids"].shape[1]
    assert torch.equal(full_enc["input_ids"][0, :n], prompt_enc["input_ids"][0]), "prompt không là tiền tố"
    labels = full_enc["input_ids"].clone()
    labels[:, :n] = -100                                     # chỉ học phần trả lời
    end = (full_enc["input_ids"][0] == IM_END).nonzero()[-1].item()
    labels[:, end + 1:] = -100                               # học tới <|im_end|>, bỏ "\n" theo sau
    item = {k: v for k, v in full_enc.items() if k in PAD_VALUES or k in ("pixel_values", "image_grid_thw")}
    unknown = set(full_enc) - set(item)
    assert not unknown, f"processor trả thêm khoá chưa xử lý: {unknown}"
    item["labels"] = labels
    return item

def collate(batch):
    L = max(b["input_ids"].shape[1] for b in batch)
    out = {k: torch.cat([F.pad(b[k], (0, L - b[k].shape[1]), value=v) for b in batch])
           for k, v in PAD_VALUES.items() if k in batch[0]}
    out["pixel_values"] = torch.cat([b["pixel_values"] for b in batch])     # nối, KHÔNG pad
    out["image_grid_thw"] = torch.cat([b["image_grid_thw"] for b in batch])
    return out

ex = encode_example(splits["train"][0])
n_img = (ex["input_ids"] == IMAGE_TOKEN).sum().item()
print("khoá:", list(ex), "| độ dài:", ex["input_ids"].shape[1], "| token ảnh:", n_img)
assert n_img == 220
print("phần được tính loss:", repr(processor.tokenizer.decode(ex["labels"][0][ex["labels"][0] != -100])))
print("đáp án            :", repr(objects_to_answer(splits["train"][0]["objects"])))
b = collate([ex, encode_example(splits["train"][1])])
print({k: tuple(v.shape) for k, v in b.items()})

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
import re
from peft import LoraConfig, PeftModel, get_peft_model
from peft.tuners.lora import LoraLayer

LLM_TARGETS = r"model\.language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
VIT_TARGETS = r"model\.visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.linear_fc[12])"
PROJECTOR_KEYS = ("visual.merger", "visual.deepstack_merger_list")
SETUPS = {"E1": dict(vit_lora=False, train_projector=False),
          "E2": dict(vit_lora=False, train_projector=True),
          "E3": dict(vit_lora=True, train_projector=True)}

def apply_setup(model, setup, r=16, alpha=32, dropout=0.05):
    s = SETUPS[setup]
    for p in model.parameters():
        p.requires_grad_(False)
    model.enable_input_require_grads()  # cần cho gradient checkpointing
    target = f"({LLM_TARGETS}|{VIT_TARGETS})" if s["vit_lora"] else LLM_TARGETS
    model = get_peft_model(model, LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout,
                                             target_modules=target, bias="none"))
    if s["train_projector"]:
        for n, p in model.named_parameters():
            if any(k in n for k in PROJECTOR_KEYS):
                p.data = p.data.float()  # fp32: AdamW/GradScaler cần master weight fp32
                p.requires_grad_(True)
    n_lora = sum(isinstance(m, LoraLayer) for m in model.modules())
    assert n_lora == (348 if s["vit_lora"] else 252), n_lora
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    proj = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and any(k in n for k in PROJECTOR_KEYS))
    print(f"{setup}: {n_lora} module LoRA | tham số huấn luyện {trainable / 1e6:.2f}M (projector {proj / 1e6:.2f}M)")
    return model, trainable

# Kiểm tra trên model "meta" (chỉ có hình dạng, không tốn RAM/VRAM): quy mô từng khối và số tham số mỗi setup
from transformers import AutoConfig
MODEL_CFG = AutoConfig.from_pretrained(MODEL_ID)
def meta_model():
    with torch.device("meta"):
        return Qwen3VLForConditionalGeneration._from_config(MODEL_CFG, dtype=torch.float16)
m = meta_model()
size = lambda pref: sum(p.numel() for n, p in m.named_parameters() if n.startswith(pref)) / 1e6
print(f"ViT {size('model.visual'):.1f}M (merger {size('model.visual.merger'):.1f}M, deepstack_merger_list "
      f"{size('model.visual.deepstack_merger_list'):.1f}M) | LLM+embedding {size('model.language_model'):.1f}M | "
      f"tổng {size(''):.1f}M (lm_head dùng chung trọng số embedding)")
TRAINABLE_META = {s: apply_setup(meta_model(), s)[1] for s in SETUPS}
print(f"E2 - E1 = {(TRAINABLE_META['E2'] - TRAINABLE_META['E1']) / 1e6:.2f}M tham số")
del m

## 6. Vòng huấn luyện
QLoRA fp16 + GradScaler, AdamW lr 2e-4, warmup 5% rồi cosine, batch 2 × tích luỹ 4, 300 bước. Mỗi 100 bước lưu adapter **và** projector (E2/E3) vào `work/runs/<eN>/`.

In [ ]:
import yaml
from torch.utils.data import DataLoader, Dataset
from transformers import get_cosine_schedule_with_warmup

CFG = {k: yaml.safe_load(open(f"configs/{k}.yaml", encoding="utf-8")) for k in ("e1", "e2", "e3")}

class TrainSet(Dataset):
    def __init__(self, entries, prompt=PROMPT, coord="norm1000"):
        self.entries, self.prompt, self.coord = entries, prompt, coord
    def __len__(self):
        return len(self.entries)
    def __getitem__(self, i):
        return encode_example(self.entries[i], self.prompt, self.coord)

def save_run(model, run_dir):
    model.save_pretrained(run_dir / "adapter")  # peft chỉ lưu LoRA, KHÔNG lưu projector
    proj = {n.removeprefix("base_model.model."): p.detach().float().cpu() for n, p in model.named_parameters()
            if p.requires_grad and any(k in n for k in PROJECTOR_KEYS)}
    if proj:
        torch.save(proj, run_dir / "projector.pt")

def train(key, cfg, entries, prompt=PROMPT, coord="norm1000", r=None, alpha=None):
    run_dir = WORK / "runs" / key
    if (run_dir / "run.json").exists():
        print(f"{key}: đã huấn luyện, dùng lại {run_dir}")
        return json.loads((run_dir / "run.json").read_text())
    run_dir.mkdir(parents=True, exist_ok=True)
    setup, lora = cfg["name"], cfg["lora"]  # "E1" | "E2" | "E3"
    r, alpha = r or lora["r"], alpha or lora["alpha"]
    max_steps, accum, bs = (4, 2, 1) if SMOKE else (cfg["max_steps"], cfg["grad_accum"], cfg["batch_size"])
    save_every = 2 if SMOKE else 100
    torch.manual_seed(cfg["seed"]); random.seed(cfg["seed"])

    model, trainable = apply_setup(load_base(), setup, r, alpha, lora["dropout"])
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, max(1, round(0.05 * max_steps)), max_steps)
    scaler = torch.amp.GradScaler("cuda")
    loader = DataLoader(TrainSet(entries, prompt, coord), batch_size=bs, shuffle=True, collate_fn=collate,
                        num_workers=2, drop_last=True, persistent_workers=True,
                        generator=torch.Generator().manual_seed(cfg["seed"]))
    def batches():
        while True:
            yield from loader
    it = batches()

    torch.cuda.reset_peak_memory_stats()
    t0, losses = time.time(), []
    for step in range(1, max_steps + 1):
        step_loss = 0.0
        for _ in range(accum):
            batch = {k: v.to(model.device) for k, v in next(it).items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**batch).loss / accum
            if not torch.isfinite(loss):
                raise RuntimeError(f"{key}: loss không hữu hạn ở bước {step} (tràn số fp16?)")
            scaler.scale(loss).backward()
            step_loss += loss.item()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        losses.append(step_loss)
        if step == 1 or step % 10 == 0:
            el = time.time() - t0
            print(f"{key} bước {step}/{max_steps} loss={step_loss:.4f} lr={sched.get_last_lr()[0]:.2e} "
                  f"{el / 60:.1f} phút (còn ~{el / step * (max_steps - step) / 60:.0f}) "
                  f"VRAM đỉnh {torch.cuda.max_memory_allocated() / 1e9:.1f}GB")
        if step % save_every == 0 or step == max_steps:
            save_run(model, run_dir)
    run = {"key": key, "setup": setup, "r": r, "alpha": alpha, "n_train_images": len(entries),
           "trainable_params": trainable,
           "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
           "train_minutes": round((time.time() - t0) / 60, 1), "steps": max_steps, "losses": losses}
    (run_dir / "run.json").write_text(json.dumps(run, indent=1))
    print(f"{key}: xong {run['train_minutes']} phút, VRAM đỉnh {run['peak_vram_gb']} GB")
    del model, opt, params, loader, it
    free_gpu()
    return run

In [ ]:
RUNS = {}
for key in ("e1", "e2", "e3"):
    try:
        RUNS[key] = train(key, CFG[key], splits["train"])
    except Exception:
        traceback.print_exc()
        print(f"!!! {key} lỗi khi huấn luyện, chuyển sang setup tiếp theo")
    free_gpu()

## 7. Suy luận và chấm E1–E3

In [ ]:
def load_finetuned(run_dir, trainable=False):
    """Model gốc 4-bit → nạp lại projector đã huấn luyện (nếu có) → gắn adapter LoRA."""
    run_dir = Path(run_dir)
    model = load_base()
    if (run_dir / "projector.pt").exists():
        sd = torch.load(run_dir / "projector.pt", map_location="cpu")
        params = dict(model.named_parameters())
        with torch.no_grad():
            for k, v in sd.items():
                params[k].copy_(v)  # KeyError nếu tên không khớp
        print(f"nạp lại {len(sd)} tensor projector ({sum(v.numel() for v in sd.values()) / 1e6:.1f}M tham số)")
    model = PeftModel.from_pretrained(model, run_dir / "adapter", is_trainable=trainable)
    return model

def eval_run(key, name, run, prompt=PROMPT, coord="norm1000"):
    model = load_finetuned(WORK / "runs" / key)
    predict(model, TEST, WORK / f"{key}.jsonl", prompt=prompt)
    del model; free_gpu()
    return score(key, name, coord=coord, meta={k: run[k] for k in ("trainable_params", "peak_vram_gb", "train_minutes")})

In [ ]:
for key in ("e1", "e2", "e3"):
    if key not in RUNS:
        continue
    try:
        RESULTS[key] = eval_run(key, key.upper(), RUNS[key])
    except Exception:
        traceback.print_exc()
        print(f"!!! {key} lỗi khi suy luận/chấm")
    free_gpu()

In [ ]:
from scripts.score import _gt_boxes, _match

def plot_losses(keys, path):
    fig, ax = plt.subplots(figsize=(8, 4))
    for k in keys:
        L = RUNS[k]["losses"]
        w = max(1, len(L) // 30)
        ax.plot([sum(L[max(0, i - w + 1):i + 1]) / len(L[max(0, i - w + 1):i + 1]) for i in range(len(L))], label=k.upper())
    ax.set_xlabel("bước tối ưu"); ax.set_ylabel("loss (trung bình trượt)"); ax.legend(); ax.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(path, dpi=110); plt.show()

def per_image(key):
    preds = {r["image_id"]: r["raw"] for r in read_jsonl(WORK / f"{key}.jsonl")}
    rows = []
    for e in TEST:
        p = parse_output(preds.get(e["image_id"])) or []
        per, _ = _match(_gt_boxes(e), p)
        tp, fp, fn = (sum(v[i] for v in per.values()) for i in range(3))
        rows.append((e, p, tp, fp, fn))
    return rows

def plot_predictions(key, path, n_good=3, n_bad=3):
    rows = [r for r in per_image(key) if r[0]["objects"] or r[1]]
    good = sorted(rows, key=lambda r: (-r[2], r[3] + r[4]))[:n_good]
    bad = sorted([r for r in rows if r not in good], key=lambda r: -(r[3] + r[4]))[:n_bad]
    fig, axes = plt.subplots(2, 3, figsize=(16, 6.5))
    for ax, (e, p, tp, fp, fn) in zip(axes.flat, good + bad):
        W, H = e["width"], e["height"]
        ax.imshow(load_image(e))
        draw_boxes(ax, gt_pixel(e), color="lime", lw=3, text=False)
        draw_boxes(ax, [(b["label"], (b["bbox"][0] * W / 1000, b["bbox"][1] * H / 1000, b["bbox"][2] * W / 1000,
                                       b["bbox"][3] * H / 1000)) for b in p], ls="--")
        ax.set_title(f"{e['image_id']}  TP={tp} FP={fp} FN={fn}", fontsize=9); ax.axis("off")
    fig.suptitle(f"{key.upper()} trên test — xanh lá: box đúng (GT); nét đứt: dự đoán "
                 "(đỏ pothole, vàng crack, xanh lơ manhole). Hàng trên: tốt nhất; hàng dưới: lỗi nhiều nhất")
    plt.tight_layout(); plt.savefig(path, dpi=110); plt.show()

if RUNS:
    plot_losses([k for k in ("e1", "e2", "e3") if k in RUNS], FIG / "loss_e1_e3.png")
plot_predictions("e0", FIG / "pred_e0.png")
for key in ("e1", "e2", "e3"):
    if key in RESULTS:
        plot_predictions(key, FIG / f"pred_{key}.png")

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# Biến chọn: data_size (10% vs 100% ảnh train), mọi thứ khác giữ như setup tốt nhất của E1–E3.
# Giá trị 100% chính là setup tốt nhất đó → chấm lại đúng file dự đoán của nó dưới tên E4.
trained = [k for k in ("e1", "e2", "e3") if k in RESULTS]
BEST = max(trained, key=lambda k: RESULTS[k]["micro"]["f1"])
print("setup tốt nhất:", BEST.upper(), {k: round(RESULTS[k]["micro"]["f1"], 3) for k in trained})

FRAC = 0.1
sub = random.Random(42).sample(splits["train"], round(FRAC * len(splits["train"])))
print(f"E4: {len(sub)} ảnh train ({FRAC:.0%}), {len({r['session'] for r in sub})} phiên")
meta_of = lambda run: {k: run[k] for k in ("trainable_params", "peak_vram_gb", "train_minutes")}
try:
    RUNS["e4_data_size_0.1"] = train("e4_data_size_0.1", CFG[BEST], sub)
    RESULTS["e4_data_size_0.1"] = eval_run("e4_data_size_0.1", f"E4 data_size=0.1 ({BEST.upper()})",
                                           RUNS["e4_data_size_0.1"])
except Exception:
    traceback.print_exc()
free_gpu()
RESULTS["e4_data_size_1.0"] = score("e4_data_size_1.0", f"E4 data_size=1.0 (= {BEST.upper()})",
                                    pred=WORK / f"{BEST}.jsonl", meta=meta_of(RUNS[BEST]))
if "e4_data_size_0.1" in RUNS:
    plot_losses([BEST, "e4_data_size_0.1"], FIG / "loss_e4.png")

In [ ]:
from scripts.score import iou_xyxy

def reward_iou(completion, gt_boxes, thr=0.5):
    """0 nếu JSON hỏng; 1 nếu đúng là không có gì; còn lại 0.2 + 0.8 * tổng IoU khớp / max(n_gt, n_pred)."""
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
    pairs = sorted(((iou_xyxy(g["bbox"], p["bbox"]), i, j) for i, g in enumerate(gt_boxes)
                    for j, p in enumerate(pred) if g["label"] == p["label"]), reverse=True)
    used_g, used_p, total = set(), set(), 0.0
    for v, i, j in pairs:
        if v < thr:
            break
        if i in used_g or j in used_p:
            continue
        used_g.add(i); used_p.add(j); total += v
    return 0.2 + 0.8 * total / max(len(gt_boxes), len(pred), 1)

# Test reward (Python thuần, không cần GPU) — viết trước khi chạy GRPO
G = [{"label": "pothole", "bbox": (100, 100, 200, 200)}]
box = lambda lab, b: {"label": lab, "bbox_2d": b}
assert reward_iou(json.dumps([box("pothole", [100, 100, 200, 200])]), G) == 1.0            # hoàn hảo
assert reward_iou('[{"label": "pothole", "bbox_2d": [100, 100', G) == 0.0                  # JSON hỏng
assert reward_iou(json.dumps([box("crack", [100, 100, 200, 200])]), G) == 0.2              # sai lớp
assert abs(reward_iou(json.dumps([box("pothole", [100, 100, 200, 200]),
                                  box("pothole", [500, 500, 600, 600])]), G) - 0.6) < 1e-9  # thừa box
assert reward_iou("[]", []) == 1.0 and reward_iou("[]", G) == 0.2                          # rỗng
assert reward_iou(json.dumps([box("pothole", [0, 0, 1000, 1000])]), G) == 0.2              # box khổng lồ
assert abs(reward_iou("```json\n" + json.dumps([box("pothole", [100, 100, 200, 210])]) + "\n```", G)
           - (0.2 + 0.8 * 100 / 110)) < 1e-9                                                 # IoU 0.909
print("OK: 7 test reward")

In [ ]:
# E5 (tuỳ chọn): GRPO khởi tạo từ adapter tốt nhất. Bọc try/except để không ảnh hưởng các phần bắt buộc.
RUN_E5 = True
if RUN_E5 and (time.time() - T_START) / 3600 > 8.5:
    RUN_E5 = False
    print("bỏ qua E5: đã chạy quá 8,5 giờ (giới hạn phiên Kaggle 12 giờ)")
if RUN_E5:
    try:
        from datasets import Dataset
        from trl import GRPOConfig, GRPOTrainer
        gcfg = yaml.safe_load(open("configs/e5.yaml", encoding="utf-8"))["grpo"]
        n_steps = 2 if SMOKE else gcfg["max_steps"]
        grpo_entries = random.Random(1).sample(splits["train"], 2 * n_steps)
        ds = Dataset.from_list([{"prompt": [{"role": "user", "content": PROMPT}], "image": load_image(e),
                                 "gt": json.dumps(_gt_boxes(e))} for e in grpo_entries])

        def grpo_reward(prompts, completions, gt, **kw):
            texts = [c[-1]["content"] if isinstance(c, list) else c for c in completions]
            texts = [t if isinstance(t, str) else "".join(x.get("text", "") for x in t) for t in texts]
            return [reward_iou(t, json.loads(g)) for t, g in zip(texts, gt)]

        e5_dir = WORK / "runs" / "e5"
        e5_dir.mkdir(parents=True, exist_ok=True)
        model = load_finetuned(WORK / "runs" / BEST, trainable=True)
        args = GRPOConfig(output_dir=str(e5_dir / "trainer"), num_generations=gcfg["num_generations"],
                          max_completion_length=gcfg["max_completion_length"], max_steps=n_steps,
                          learning_rate=gcfg["learning_rate"], beta=gcfg["beta"], use_vllm=False,
                          per_device_train_batch_size=4, gradient_accumulation_steps=2,
                          gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
                          fp16=False, bf16=False, logging_steps=1 if SMOKE else 5, save_strategy="no",
                          report_to="none", seed=42, log_completions=False)
        trainer = GRPOTrainer(model=model, reward_funcs=grpo_reward, args=args, train_dataset=ds,
                              processing_class=processor)
        for p in trainer.model.parameters():  # trl đưa adapter của model 4-bit về bf16; T4 không có bf16
            if p.requires_grad:
                p.data = p.data.float()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        trainer.train()
        e5_run = {"trainable_params": sum(p.numel() for p in trainer.model.parameters() if p.requires_grad),
                  "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
                  "train_minutes": round((time.time() - t0) / 60, 1), "init_from": BEST,
                  "log_history": trainer.state.log_history}
        trainer.model.save_pretrained(e5_dir / "adapter")
        if (WORK / "runs" / BEST / "projector.pt").exists():
            shutil.copy(WORK / "runs" / BEST / "projector.pt", e5_dir / "projector.pt")
        (e5_dir / "run.json").write_text(json.dumps(e5_run, indent=1, default=str))
        del trainer, model; free_gpu()

        RESULTS["e5"] = eval_run("e5", f"E5 GRPO (từ {BEST.upper()})", e5_run)
        logs = [h for h in e5_run["log_history"] if "reward" in h]
        if logs:
            fig, ax = plt.subplots(figsize=(7, 3.5))
            ax.plot([h["step"] for h in logs], [h["reward"] for h in logs], marker="o")
            ax.set_xlabel("bước GRPO"); ax.set_ylabel("reward trung bình"); ax.grid(alpha=.3)
            plt.tight_layout(); plt.savefig(FIG / "e5_reward.png", dpi=110); plt.show()
        # Dấu hiệu reward hacking: trả [] hoặc box khổng lồ nhiều hơn SFT?
        for k in (BEST, "e5"):
            ps = [parse_output(r["raw"]) for r in read_jsonl(WORK / f"{k}.jsonl")]
            bs = [b for p in ps if p for b in p]
            big = sum((b["bbox"][2] - b["bbox"][0]) * (b["bbox"][3] - b["bbox"][1]) > 0.25e6 for b in bs)
            print(f"{k}: trả [] {sum(p == [] for p in ps)}/{len(ps)} | box chiếm >25% ảnh {big}/{len(bs)} | "
                  f"số box TB/ảnh {len(bs) / len(ps):.2f}")
    except Exception:
        traceback.print_exc()
        print("!!! E5 lỗi (phần tuỳ chọn) — các kết quả bắt buộc không bị ảnh hưởng")
    free_gpu()

## Tổng hợp

In [ ]:
import pandas as pd
rows = []
for p in sorted(EVID.glob("*.json")):
    d = json.loads(p.read_text(encoding="utf-8"))
    rows.append({"file": p.name, "F1": d["micro"]["f1"], "CI95": "%.3f–%.3f" % tuple(d["f1_ci95"]),
                 "P": d["micro"]["precision"], "R": d["micro"]["recall"], "valid": d["parse_valid_rate"],
                 "FP/ảnh": d["fp_per_image"], "meanIoU": d["mean_iou"],
                 **{f"F1 {c}": d["per_class"][c]["f1"] for c in CLASSES},
                 **{k: d["meta"].get(k) for k in ("trainable_params", "peak_vram_gb", "train_minutes")}})
summary = pd.DataFrame(rows).set_index("file")
summary.to_csv(WORK / "summary.csv")
print(f"tổng thời gian notebook: {(time.time() - T_START) / 3600:.2f} giờ")
summary.round(3)

In [ ]:
print(subprocess.run([sys.executable, "scripts/check_submission.py"], capture_output=True, text=True).stdout)
if IS_KAGGLE:
    shutil.rmtree(DATA / "images", ignore_errors=True)  # ảnh gốc không cần trong output của Kaggle
missing = [k for k in ("e0", "e1", "e2", "e3", "e4_data_size_0.1", "e4_data_size_1.0") if k not in RESULTS]
if missing and not SMOKE:
    raise RuntimeError(f"thiếu kết quả: {missing} — xem log lỗi phía trên")
print("Xong. Tải về: report/evidence/*.json, report/figures/*.png, work/*.jsonl, work/runs/*/run.json")